In [1]:
from gurobipy import Model, GRB, quicksum

# Sets
nurses=list(range(1, 5))
patients=list(range(1, 21))
services=[1,2]
nodes=[0]+patients
advance_patients=list(range(1, 19))
dynamic_patients=list(range(19, 21))

# Parameters

import random
import math
random.seed(27)

# Coordinates for depot + patients
coords = {}

# Depot
coords[0] = (25, 25)

# Patients
for i in range(1, 21):
    coords[i] = (random.uniform(0, 50), random.uniform(0, 50))

# Travel time matrix (symmetric)
travel_time = {}

speed = 1.0  # distance units per minute (scale if needed)

for i in range(21):
    for j in range(21):
        if i != j:
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            dist = math.hypot(dx, dy)
            travel_time[(i, j)] = dist / speed

#service_duration = {1: 30, 2: 45, 3: 30, 4:20, 5:60}
#service_duration = {1: 30, 2: 45, 3: 60, 4:20}
service_duration = {1: 30, 2: 45}

request = {i: [random.randint(1, 2)] for i in range(1, 21)}

sync_patients = random.sample(patients, 2)
sync = {i: int(i in sync_patients) for i in patients}


priority = {i: random.randint(3, 5) for i in range(1, 21)}


patient_time_windows = {}

for i in range(1, 19):  # Only advance patients
    start = random.randrange(0, 301, 15)  # multiples of 15 between 0 and 300
    end = start + 180                     # fixed 3-hour window
    patient_time_windows[i] = (start, end)


nurse_time_windows = {i: (0, 480) for i in range(1, 5)}

qual = {}


# Assign 2 nurses per service
for s in services:
    assigned_nurses = nurses[(s - 1) * 2 : s * 2]
    for n in assigned_nurses:
        for s2 in services:
            qual[(n, s2)] = int(s == s2)

request_time = {}
for i in range(19, 21):
    request_time[i] = random.randint(0, 300)


# Create Model
m = Model("HHC_Scheduling")

# Decision variables
x = m.addVars(nodes, nodes, nurses, vtype=GRB.BINARY, name="x")  # travel
z = m.addVars(patients, nurses, services, vtype=GRB.BINARY, name="z")  # service assigned
S = m.addVars(patients, nurses, vtype=GRB.CONTINUOUS, name="S")  # start time
du = m.addVars(dynamic_patients, vtype=GRB.CONTINUOUS, name="du")
cover = m.addVars([(i,s) for i in patients for s in request[i]],vtype=GRB.BINARY, name="cover")

# Objective: weighted combination
cD, cT, cW, cU = 1.0, 1.0, 1.0, 1000.0

lag_term = quicksum(du[i] for i in dynamic_patients)

travel_term = quicksum(travel_time[i, j] * x[i, j, k]
                       for i in nodes for j in nodes if i != j for k in nurses if (i, j) in travel_time)

#wait_term=quicksum(w[i,j,k] for i in patients for j in patients for k in nurses)

#coverage_term = quicksum(z[i, k, s] for i in patients for s in services for k in nurses)
coverage_term = quicksum(priority[i]*(1-cover[i,s]) for i in patients for s in request[i])
#m.setObjective(cD * lag_term + cT * travel_term + cW * wait_term + cU * coverage_term, GRB.MINIMIZE)
m.setObjective(cD * lag_term + cT * travel_term + cU * coverage_term, GRB.MINIMIZE)
# Constraints
M = 1e5
# Tour construction
for k in nurses:
    m.addConstr(quicksum(x[0, j, k] for j in patients) <= 1)
    m.addConstr(quicksum(x[j, 0, k] for j in patients) <= 1)

# Continuity of tour    
for k in nurses:
    for j in nodes:
        m.addConstr(
            quicksum(x[i, j, k] for i in nodes if i != j) ==
            quicksum(x[j, l, k] for l in nodes if l != j)
            )
        
# Dynamic patients
for i in dynamic_patients:
    for k in nurses:
        for s in services:
            if s in request[i]:
                m.addConstr(du[i] >= S[i, k] - request_time[i] - M * (1 - z[i, k, s]))
                m.addConstr(S[i, k] >= request_time[i] * z[i, k, s])
                m.addConstr(S[i, k] <= M * z[i, k, s])

# Patient time window
for i in advance_patients:  # only for patients with known requests
    E_i, L_i = patient_time_windows[i]
    for k in nurses:
        lhs = quicksum(x[j, i, k] for j in nodes if j != i)
        m.addConstr(E_i * lhs <= S[i, k])
        m.addConstr(S[i, k] <= L_i * lhs)

# Nurse time window
for k in nurses:
    et_k = nurse_time_windows[k][0]
    for j in patients:
        m.addConstr(S[j, k] >= et_k + travel_time[0, j]- M * (1 - x[0, j, k]))

for j in patients:
    for k in nurses:
        lt_k = nurse_time_windows[k][1]
        service_time = quicksum(service_duration[s] * z[j, k, s] for s in services)
        m.addConstr(
            S[j, k] + service_time <= lt_k - travel_time[j, 0] + M * (1 - x[j, 0, k])
        )

# Qualification
for k in nurses:
    for s in services:
        m.addConstr(
            quicksum(z[i,k,s] for i in patients) <= M*qual[k,s]
        )

# Time of entrance and exit
for i in patients:
    for j in patients:
        #if i == j:
        #    continue
        for k in nurses:
            if (i, j) in travel_time:
                travel = travel_time[i, j]
                service_time = quicksum(service_duration[s] * z[i, k, s] for s in services)
                m.addConstr(
                    S[i, k] + travel + service_time <= S[j, k] + M * (1 - x[i, j, k])
                )

# Synchronization
for i in patients:
    for s in request[i]:
        m.addConstr(quicksum(z[i, k, s] for k in nurses) == (1 + sync[i]) * cover[i, s])

        
for i in patients:
    for s in services:
        if sync[i] == 1 and s in request[i]:
            for k1 in nurses:
                for k2 in nurses:
                    if k1 != k2:
                        m.addConstr(
                            S[i, k1] - S[i, k2] <= M * (2 - z[i, k1, s] - z[i, k2, s])
                        )
                        m.addConstr(
                            S[i, k2] - S[i, k1] <= M * (2 - z[i, k1, s] - z[i, k2, s])
                        )

# Relations among variables
for j in patients:
    for k in nurses:
        incoming = quicksum(x[i, j, k] for i in nodes if i != j)
        service_assigned = quicksum(z[j, k, s] for s in services if s in request[j])

        # Constraint 1: no service if not visited
        m.addConstr(service_assigned <= M * incoming)

        # Constraint 2: no visit if no service
        m.addConstr(incoming <= service_assigned)

        
###################################### Should be reconsidered

for i in patients:
    for k in nurses:
        for s in services:
            if s not in request[i]:
                m.addConstr(z[i, k, s] == 0)

#m.setParam("TimeLimit", 7200) 
#m.Params.MIPGap = 0.9 
m.optimize()
'''
if m.SolCount > 0:
    print("Best objective:", m.ObjVal)
    print("Best bound:", m.ObjBound)
    print("Relative gap:", m.MIPGap)
else:
    print("No feasible solution found.")
'''
if m.status == GRB.OPTIMAL:
    for i in patients:
        for s in services:
            for k in nurses:
                if z[i, k, s].X > 0.5:
                    print(f"Nurse {k} serves Patient {i} (Service {s}) at time {S[i, k].X:.1f}")
                    print
else:
    print("No optimal solution found.")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2692963
Academic license 2692963 - for non-commercial use only - registered to ba___@usc.edu
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Snapdragon X Plus (8-core) @ 3.30 GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2692963 - for non-commercial use only - registered to ba___@usc.edu
Optimize a model with 2256 rows, 2026 columns and 18412 nonzeros (Min)
Model fingerprint: 0xb68dc05d
Model has 1702 linear objective coefficients and an objective constant of 80000
Variable types: 82 continuous, 1944 integer (1944 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+05]
  Objective range  [1e+00, 5e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+05]

Found heuristic solution: objective 80000.000000
Presolve removed 1959 rows and 1773 columns
Pre

In [2]:
# After m.optimize():
nurse_routes = {k: [] for k in nurses}
for k in nurses:
    current = 0
    visited = set()
    while True:
        next_node = next((j for j in nodes if j != current
                          and (current, j) not in visited
                          and x[current, j, k].X > 0.5), None)
        if next_node is None:
            break
        nurse_routes[k].append(next_node)
        visited.add((current, next_node))
        current = next_node
    nurse_routes[k] = [0] + nurse_routes[k]

In [3]:
lag_term.getValue()

0.0

In [4]:
travel_term.getValue()

448.81256040396767

In [5]:
coverage_term.getValue()

0.0